# Load Dataset

In [4]:
import pandas as pd

In [5]:
df= pd.read_csv("/content/gk_qna_dataset.csv")

In [6]:
df

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo
...,...,...
166,"Can you tell me, ""Which bird cannot fly""?",Ostrich
167,"I was wondering, ""What is 'H2O' commonly called""?",Water
168,"I was wondering, ""Which planet is known as the...",Mars
169,"Can you tell me, ""What is the main religion in...",Islam


In [7]:
df.head()

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo


In [8]:
df["question"][0]

'What is the capital of "France"?'

# tokenize


In [9]:
import re

def tokenize(text):

  #1  Lowercase
  text = text.lower()

  #2 remove quotes
  text = re.sub( r"[\"']","",text)


  #3 remove all punctuations
  text = re.sub(r"[^a-z0-9\s]", " ", text)

  #4 remove extra space
  text = re.sub(r"\s+", " ", text).strip()

  #tokenize split into words
  tokens = text.split()

  return tokens

In [10]:
print(tokenize(df["question"][0]))

['what', 'is', 'the', 'capital', 'of', 'france']


# Vocab forming

In [11]:
vocab = {'<UNK>':0}

In [23]:
def build_vocab(row):
  print(row["question"], row["answer"])


  tokenized_question= tokenize(row["question"])
  tokenized_answer= tokenize(row["answer"])

  #print(tokeniz_question, tokeniz_answer)

  merged_tokens = tokenized_question + tokenized_answer
  # print(merged_tokens)

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)


In [24]:
df.apply(build_vocab, axis=1)

What is the capital of "France"? Paris
What is the capital of "Germany"? Berlin
What is the capital of "Italy"? Rome
What is the capital of "Spain"? Madrid
What is the capital of "Japan"? Tokyo
What is the capital of "Canada"? Ottawa
What is the capital of "Brazil"? Brasilia
What is the capital of "Australia"? Canberra
What is the capital of "India"? New Delhi
What is the capital of "China"? Beijing
What is the capital of "Russia"? Moscow
What is the capital of "United States"? Washington DC
What is the capital of "Mexico"? Mexico City
What is the capital of "Egypt"? Cairo
What is the capital of "Turkey"? Ankara
What is the capital of "Argentina"? Buenos Aires
What is the capital of "South Korea"? Seoul
What is the capital of "Indonesia"? Jakarta
What is the capital of "Pakistan"? Islamabad
What is the capital of "Bangladesh"? Dhaka
What is the capital of "Nepal"? Kathmandu
What is the capital of "Sri Lanka"? Colombo
What is the capital of "Thailand"? Bangkok
What is the capital of "Ma

,0
0,None
1,None
2,None
3,None
4,None
...,...
166,None
167,None
168,None
169,None


In [25]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'italy': 10,
 'rome': 11,
 'spain': 12,
 'madrid': 13,
 'japan': 14,
 'tokyo': 15,
 'canada': 16,
 'ottawa': 17,
 'brazil': 18,
 'brasilia': 19,
 'australia': 20,
 'canberra': 21,
 'india': 22,
 'new': 23,
 'delhi': 24,
 'china': 25,
 'beijing': 26,
 'russia': 27,
 'moscow': 28,
 'united': 29,
 'states': 30,
 'washington': 31,
 'dc': 32,
 'mexico': 33,
 'city': 34,
 'egypt': 35,
 'cairo': 36,
 'turkey': 37,
 'ankara': 38,
 'argentina': 39,
 'buenos': 40,
 'aires': 41,
 'south': 42,
 'korea': 43,
 'seoul': 44,
 'indonesia': 45,
 'jakarta': 46,
 'pakistan': 47,
 'islamabad': 48,
 'bangladesh': 49,
 'dhaka': 50,
 'nepal': 51,
 'kathmandu': 52,
 'sri': 53,
 'lanka': 54,
 'colombo': 55,
 'thailand': 56,
 'bangkok': 57,
 'malaysia': 58,
 'kuala': 59,
 'lumpur': 60,
 'vietnam': 61,
 'hanoi': 62,
 'uae': 63,
 'abu': 64,
 'dhabi': 65,
 'iran': 66,
 'tehran': 67,
 'iraq

In [26]:
len(vocab)

243

In [27]:
vocab['what']

1

# text to indices

In [28]:
def text_to_indices(text,vocab):
  indexed_text = []

  for token in tokenize(text):
    # print(token)
    if token in vocab:
      # print(vocab[token])
      indexed_text.append(vocab[token])

    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text


In [29]:
text_to_indices("the is Phitron",vocab)

[3, 2, 0]

# Dataset and Dataloader

In [30]:
import torch
from torch.utils.data import Dataset,DataLoader

In [31]:
df.shape[0]

171

In [32]:
index = 0

text_to_indices(df.iloc[index]['question'],vocab)

[1, 2, 3, 4, 5, 6]

In [33]:
class QADataset(Dataset):

  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [34]:
dataset = QADataset(df,vocab)

In [35]:
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [36]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [37]:
for question,answer in dataloader:
  print(question,answer[0])

tensor([[237,   1, 238, 239,   1, 131, 145, 133, 121]]) tensor([146, 147, 148])
tensor([[ 1,  2,  3,  4,  5, 12]]) tensor([13])
tensor([[ 95, 232, 233,  79,  84,   3, 161, 162]]) tensor([163, 164])
tensor([[240, 232, 241, 242,  93,  94,  95, 206, 207]]) tensor([208])
tensor([[  1,   2,   3, 111, 112]]) tensor([113, 112])
tensor([[ 1,  2,  3,  4,  5, 20]]) tensor([21])
tensor([[237,   1, 238, 239,   1,   2, 136]]) tensor([137, 138])
tensor([[  1,   2, 124, 120, 121]]) tensor([125, 126])
tensor([[  1, 131, 132, 133, 121]]) tensor([134, 130, 135])
tensor([[230, 231,   1,   2,   3, 220, 129,   5,  12]]) tensor([222])
tensor([[240, 232, 241, 242,  93, 197,   2, 200]]) tensor([201])
tensor([[ 1,  2,  3,  4,  5, 45]]) tensor([46])
tensor([[234, 235, 236,  93, 100,   2, 101, 102,   3, 103, 100]]) tensor([104])
tensor([[ 1,  2,  3,  4,  5, 75]]) tensor([76])
tensor([[93, 94, 95, 96, 97]]) tensor([98, 99])
tensor([[230, 231,   1, 131, 139, 133, 121]]) tensor([140, 130, 135])
tensor([[234, 235, 2

In [38]:
# squeeze

import torch

x = torch.tensor( [[[1,2,3]]] )

# print(x.shape)

y = x.squeeze(0)

print(y.shape)

torch.Size([1, 3])


# RNN Architecture Implementation

In [39]:
import torch.nn as nn

class simpleRNN(nn.Module):
  def __init__(self,vocab_size,embedding_dim=50,hidden_size=64):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size,embedding_dim)
    self.rnn = nn.RNN(embedding_dim,hidden_size,batch_first=True)
    self.fc = nn.Linear(hidden_size,vocab_size)


  def forward(self,question):
     embedded = self.embedding(question) # (1, seq_len, 50)
     _,final = self.rnn(embedded)

     return self.fc(final.squeeze(0))  # (1, vocab_size)



In [40]:
model = simpleRNN(vocab_size = len(vocab))

In [41]:
# EXample of unsqueeze

import torch

x = torch.tensor([1, 2, 3])   # shape: (3,)
print(x.shape)

y = x.unsqueeze(0)
print(y.shape)

torch.Size([3])
torch.Size([1, 3])


# training loop

In [42]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
epochs = 50

In [43]:
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
        optimizer.zero_grad()
        output = model(question)
        # Fix: take only the first answer token as target → shape (1,)
        target = answer[0][0].unsqueeze(0)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs}  |  Loss: {total_loss:.4f}")


Epoch  10/50  |  Loss: 592.2536
Epoch  20/50  |  Loss: 364.2504
Epoch  30/50  |  Loss: 243.4738
Epoch  40/50  |  Loss: 173.5436
Epoch  50/50  |  Loss: 124.1299


# Prediction

In [44]:
vocab.items()

dict_items([('<UNK>', 0), ('what', 1), ('is', 2), ('the', 3), ('capital', 4), ('of', 5), ('france', 6), ('paris', 7), ('germany', 8), ('berlin', 9), ('italy', 10), ('rome', 11), ('spain', 12), ('madrid', 13), ('japan', 14), ('tokyo', 15), ('canada', 16), ('ottawa', 17), ('brazil', 18), ('brasilia', 19), ('australia', 20), ('canberra', 21), ('india', 22), ('new', 23), ('delhi', 24), ('china', 25), ('beijing', 26), ('russia', 27), ('moscow', 28), ('united', 29), ('states', 30), ('washington', 31), ('dc', 32), ('mexico', 33), ('city', 34), ('egypt', 35), ('cairo', 36), ('turkey', 37), ('ankara', 38), ('argentina', 39), ('buenos', 40), ('aires', 41), ('south', 42), ('korea', 43), ('seoul', 44), ('indonesia', 45), ('jakarta', 46), ('pakistan', 47), ('islamabad', 48), ('bangladesh', 49), ('dhaka', 50), ('nepal', 51), ('kathmandu', 52), ('sri', 53), ('lanka', 54), ('colombo', 55), ('thailand', 56), ('bangkok', 57), ('malaysia', 58), ('kuala', 59), ('lumpur', 60), ('vietnam', 61), ('hanoi', 62

In [45]:
idx_to_word = { idx : word for word,idx in vocab.items()  }
idx_to_word

{0: '<UNK>',
 1: 'what',
 2: 'is',
 3: 'the',
 4: 'capital',
 5: 'of',
 6: 'france',
 7: 'paris',
 8: 'germany',
 9: 'berlin',
 10: 'italy',
 11: 'rome',
 12: 'spain',
 13: 'madrid',
 14: 'japan',
 15: 'tokyo',
 16: 'canada',
 17: 'ottawa',
 18: 'brazil',
 19: 'brasilia',
 20: 'australia',
 21: 'canberra',
 22: 'india',
 23: 'new',
 24: 'delhi',
 25: 'china',
 26: 'beijing',
 27: 'russia',
 28: 'moscow',
 29: 'united',
 30: 'states',
 31: 'washington',
 32: 'dc',
 33: 'mexico',
 34: 'city',
 35: 'egypt',
 36: 'cairo',
 37: 'turkey',
 38: 'ankara',
 39: 'argentina',
 40: 'buenos',
 41: 'aires',
 42: 'south',
 43: 'korea',
 44: 'seoul',
 45: 'indonesia',
 46: 'jakarta',
 47: 'pakistan',
 48: 'islamabad',
 49: 'bangladesh',
 50: 'dhaka',
 51: 'nepal',
 52: 'kathmandu',
 53: 'sri',
 54: 'lanka',
 55: 'colombo',
 56: 'thailand',
 57: 'bangkok',
 58: 'malaysia',
 59: 'kuala',
 60: 'lumpur',
 61: 'vietnam',
 62: 'hanoi',
 63: 'uae',
 64: 'abu',
 65: 'dhabi',
 66: 'iran',
 67: 'tehran',
 68: '

In [46]:
def predict(question_text,model,vocab,idx_to_word):
  model.eval()

  with torch.no_grad():
    indices = text_to_indices(question_text,vocab)

    if not indices:
      return "<UNK>"

    x = torch.tensor(indices).unsqueeze(0)  #batch size, sequence legth

    pred = torch.argmax(model(x),dim=1).item()


  return idx_to_word.get(pred,"<UNK>")

In [47]:
# Test
tests = [
    "What is the capital of France?",
    "What is H2O commonly called?",
    "Which planet is known as the red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",

]
for q in tests:
    print(f"{q:45} → {predict(q, model, vocab, idx_to_word)}")

What is the capital of France?                → paris
What is H2O commonly called?                  → water
Which planet is known as the red planet?      → mars
Which bird cannot fly?                        → ostrich
What is the capital of Japan ?                → tokyo
